# 01 · Prompts → Ollama → respuestas → preparación del sandbox

1. Resolver la raíz e importar las funciones del proyecto.
2. Exportar o reutilizar los prompts TACO en JSONL.
3. Cargar una muestra y enviarla a Ollama.
4. Guardar respuestas incrementalmente y preparar tareas para un futuro sandbox.

El texto original del problema se conserva. Las instrucciones de Python se
mandan aparte como `system`. **No se ejecuta el código que devuelve el modelo.**


## 1. Entorno e imports

Primero localizamos el repositorio para que los imports funcionen tanto desde su raíz como desde la carpeta del notebook. No hay imports con `*` ni lógica de inferencia escondida en las celdas.

In [1]:
# Biblioteca estándar: rutas y acceso a módulos del proyecto.
from pathlib import Path
import sys

working_directory = Path.cwd().resolve()
PROJECT_ROOT = None
for directory in (working_directory, *working_directory.parents):
    if (directory / "modeling/src/data/extract_prompts.py").is_file():
        PROJECT_ROOT = directory
        break
if PROJECT_ROOT is None:
    raise RuntimeError("Abre el notebook dentro del repositorio prompt-difficulty-predictor")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [2]:
# Biblioteca estándar: identificador de la ejecución y vista previa.
from itertools import islice

# Proyecto: un adaptador, un evaluador y utilidades de archivos.
from backend.local_llm.ollama_adapter import OllamaAdapter
from modeling.src.data.extract_prompts import export_prompts, iter_questions
from modeling.src.data.prompt_io import load_prompts, read_jsonl
from modeling.src.evaluation.batch import CODING_SYSTEM, generate_batch
from modeling.src.evaluation.generate_responses import EvaluationRunner
from modeling.src.sandbox.contracts import SandboxLimits
from modeling.src.sandbox.prepare import prepare_sandbox_tasks


## 2. Configuración

`MAX_PROMPTS=3` permite una primera prueba pequeña. `RUN_OLLAMA=False` evita
relanzar inferencias al ejecutar todas las celdas automáticamente. Cámbialo a
`True` para generar. Usa un modelo instalado en tu Ollama.

Para reanudar, conserva `RUN_ID`; para cambiar modelo, opciones o instrucciones,
usa un nuevo identificador. Las generaciones completadas no se repiten y los
errores se vuelven a intentar. No hay ejecución concurrente sobre una misma carpeta.


In [3]:
DATA_DIR = PROJECT_ROOT / "modeling/data"
PROMPTS_PATH = DATA_DIR / "processed/prompts.jsonl"
RAW_TACO_DIR = DATA_DIR / "raw/TACO"
REBUILD_PROMPTS = False

OLLAMA_HOST = "http://localhost:11434"
MODEL = "qwen3:8b" # Cambiar a llama3.1:8b
OPTIONS = {"temperature": 0.0, "seed": 42, "num_ctx": 4096, "num_predict": 512}
MAX_PROMPTS = 3
RUN_OLLAMA = False

# Este ID contiene el piloto real validado. Cambiarlo para un experimento nuevo.
RUN_ID = "pilot_20261026_qwen3_notebook"
RUN_DIR = PROJECT_ROOT / "modeling/experiments/runs" / RUN_ID
RESPONSES_PATH = RUN_DIR / "responses.jsonl"
TASKS_PATH = RUN_DIR / "sandbox_tasks.jsonl"


## 3. Obtener y cargar prompts

El archivo de origen conserva solo `{"prompt": "..."}`. Al cargarlo se calcula
un `prompt_id` SHA-256 estable para enlazar respuestas y pruebas. La carga es
progresiva: no convierte los 25,816 registros en una lista ni necesita pandas.


In [4]:
if REBUILD_PROMPTS or not PROMPTS_PATH.exists():
    summary = export_prompts(iter_questions(RAW_TACO_DIR, ["train", "test"]), PROMPTS_PATH)
    print(summary)
else:
    print(f"Reutilizando: {PROMPTS_PATH}")

# Valida todos los registros sin mantenerlos en memoria.
total_prompts = sum(1 for _ in load_prompts(PROMPTS_PATH))
preview = list(load_prompts(PROMPTS_PATH, limit=min(MAX_PROMPTS, 3)))
print(f"Prompts válidos: {total_prompts:,}")
for item in preview:
    print(item["prompt_id"][:12], item["prompt"][:160].replace("\n", " "))


Reutilizando: /Users/emiliogarcialopez/projectGitHub/deep_learning_Project/prompt-difficulty-predictor/modeling/data/processed/prompts.jsonl
Prompts válidos: 25,816
e0e1a40c9f75 This is an interactive problem.  In good old times dwarves tried to develop extrasensory abilities:    * Exactly n dwarves entered completely dark cave.    * Ea
f6e6e2933845 There are $n$ candy boxes in front of Tania. The boxes are arranged in a row from left to right, numbered from $1$ to $n$. The $i$-th box contains $r_i$ candies
b024a891ac3b Little Petya likes to play a lot. Most of all he likes to play a game «Holes». This is a game for one person with following rules:  There are N holes located in


## 4. Generar y guardar respuestas

`responses.jsonl` guarda inmediatamente cada intento: IDs, prompt, texto de
respuesta, modelo, opciones, estado, fecha y tiempos. `config.json` conserva
las instrucciones y configuración de la ejecución. No se pierde todo el lote
si se interrumpe después de una respuesta. Una línea incompleta por un cierre
abrupto se reporta como error y debe revisarse antes de reanudar.

`think=False` desactiva el razonamiento separado en los modelos que lo admiten.
Aumenta `num_ctx` para problemas largos; el tamaño actual no garantiza que todos
los enunciados del dataset quepan completos. `done_reason=length` señala una
salida que agotó el presupuesto y no se considera lista para el sandbox.


In [5]:
if RUN_OLLAMA:
    adapter = OllamaAdapter(
        MODEL, host=OLLAMA_HOST, timeout=300, options=OPTIONS,
        system=CODING_SYSTEM, think=False,
    )
    evaluator = EvaluationRunner(adapter)
    summary = generate_batch(
        load_prompts(PROMPTS_PATH, limit=MAX_PROMPTS), evaluator, RUN_DIR,
    )
    print(summary)
else:
    print("Inferencia desactivada. Define RUN_OLLAMA=True para enviar la muestra a Ollama.")


Inferencia desactivada. Define RUN_OLLAMA=True para enviar la muestra a Ollama.


In [6]:
if RESPONSES_PATH.exists():
    for record in islice(read_jsonl(RESPONSES_PATH), 3):
        print(record["prompt_id"][:12], record["status"], record.get("done_reason"))
        print(record.get("response", record.get("error", ""))[:400])
else:
    print("Aún no hay respuestas para este RUN_ID.")


e0e1a40c9f75 generated stop
```python
import sys
import random

def main():
    import sys
    input = sys.stdin.readline

    n = int(input())
    points = []
    for _ in range(n):
        x = random.randint(0, 10**9)
        y = random.randint(0, 10**9)
        while (x, y) in points:
            x = random.randint(0, 10**9)
            y = random.randint(0, 10**9)
        print(x, y)
        sys.stdout.flush()
        co
f6e6e2933845 generated stop
```python
import sys
import heapq

def main():
    import sys
    input = sys.stdin.read
    data = input().split()
    
    n = int(data[0])
    s = int(data[1])
    k = int(data[2])
    
    r = list(map(int, data[3:3+n]))
    c = data[3+n]
    
    s -= 1  # convert to 0-based index
    
    # dp[i][j][last_color][last_count] = min_time
    # i: number of eaten candies
    # j: current box posi
b024a891ac3b generated stop
```python
import sys
input = sys.stdin.read

def main():
    data = input().split()
    idx = 0
    N = int(data[i

## 5. Preparar el sandbox (sin ejecutar código)

Cada tarea enlaza `prompt_id` y `response_id`, extrae un único bloque Python
y adjunta límites propuestos. Estados: `pending_test_spec`, `needs_code_review`
o `incomplete_generation`; los problemas interactivos se separan como
`requires_interactive_judge`. No son resultados de corrección.

Falta conectar los tests reales de TACO mediante un índice `prompt_id → split/idx`
y definir su modalidad (`stdin` o función). No se usan los ejemplos del prompt
como sustituto de esos tests. Después implementaremos `SandboxRunner` en un
worker aislado, fuera del kernel de Jupyter y del proceso que llama a Ollama.


In [7]:
if RESPONSES_PATH.exists():
    limits = SandboxLimits(wall_time_seconds=5, memory_mb=256)
    print(prepare_sandbox_tasks(RESPONSES_PATH, TASKS_PATH, limits=limits))
else:
    print("Primero genera respuestas o selecciona un RUN_ID existente.")


{'requires_interactive_judge': 1, 'pending_test_spec': 2, 'tasks_path': '/Users/emiliogarcialopez/projectGitHub/deep_learning_Project/prompt-difficulty-predictor/modeling/experiments/runs/pilot_20261026_qwen3_notebook/sandbox_tasks.jsonl'}


## Archivos de esta ejecución

```text
modeling/experiments/runs/<RUN_ID>/
├── config.json           # configuración reproducible de solicitudes
├── responses.jsonl       # respuestas e intentos incrementales
└── sandbox_tasks.jsonl   # tareas preparadas; no ejecutadas
```

La implementación del sandbox irá en `modeling/src/sandbox/`. Este notebook
solo prepara los archivos. Consulta `modeling/src/sandbox/README.md`.
